# Experiment 9b — Option B smoke: deterministic lookup conflict, no word-order divergence

Grammar locked from the sanity run (LEVELS=6, NT=4, T=4, degree=4; d=96 monolingual acc=0.554).

## What this notebook does differently from the last smoke
1. **L_B has the same word order as L_A.** The only thing that differs between languages is a
   context-keyed terminal lookup. No reversal anywhere.
2. **Conflict is deterministic.** Every terminal's rendering is a pure function of its tree
   context `(grandparent NT, parent NT, position in rule)`. No per-draw coin flips, so there is
   no irreducible noise. A conflicted context re-maps terminals with a fixed derangement for
   L_B; L_A is always identity.
3. **rho = fraction of contexts whose rendering agrees with the anchor.** rho=1 is the control
   (L_B identical to L_A apart from its BOS tag). Conflict sets are nested across rho.
4. **Eval fixes:** padding is masked by an explicit validity mask (the old `y != 0` mask silently
   dropped every real terminal-0 target), and principal angles are computed between
   feature-space subspaces (right singular directions, Chang-style), not sample-space ones.

## Measurements (held-out trees, same trees rendered in both languages)
- CE per language, split by **conflicted vs clean target positions**.
- Within-model principal angle (k=4, mean-centered) between A-run and B-run hidden states at
  matched positions, split conflicted/clean, at a mid layer and the last layer
  (the last layer separates trivially from output routing, so read mid first).
- Always compared against a **rho=1 control model** trained with identical data order and init.

## Grid (smoke)
d in {8, 32, 96} x {control, key-depth 1 @ rho=0.5, key-depth 2 @ rho=0.5} = 9 models, ~25 min.
Key depth 1 = (parent, position) contexts (~10 lookups); depth 2 adds the grandparent (~40).
Two depths because we do not know how many lookups it takes to bind capacity.

In [1]:
import os, time, math, random, itertools, statistics as _st
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext

GPU_ID = 1
DEVICE = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")
print(f"using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  {torch.cuda.get_device_name(GPU_ID)}")

def autocast_ctx():
    return (torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16)
            if DEVICE.type == "cuda" else nullcontext())

torch.manual_seed(0)
random.seed(0)

using device: cuda:1
  NVIDIA H200 NVL


## 1. Config

In [2]:
# ---- grammar (locked from sanity) ----
LEVELS       = 6
NT_SIZE      = 4
TERM_SIZE    = 4
MIN_DEGREE   = 4
MAX_DEGREE   = 4
RULE_LEN_CHOICES = [2, 3]
MAXK         = max(RULE_LEN_CHOICES)

# ---- two languages, SAME word order (Option B) ----
BOS        = {"A": TERM_SIZE, "B": TERM_SIZE + 1}
VOCAB_SIZE = TERM_SIZE + 2
MIX_A      = 0.70                       # 70% anchor / 30% L_B per batch

# ---- conflict ----
RHO        = 0.5                        # fraction of contexts that AGREE with the anchor
KEY_DEPTHS = [1, 2]                     # 1: (parent, pos)   2: (grandparent, parent, pos)

# ---- smoke grid ----  (full: D_LIST [8,16,32,96], SEEDS [0,7], N_B [1500,12000],
#                        RHO in [0.85,0.7,0.5] with ref set = lowest-rho set)
D_LIST     = [8, 32, 96]
SEEDS      = [0]
N_B        = 12000                      # trees in L_B's bounded pool (data budget); anchor is online
N_EVAL     = 300

# ---- training / eval ----
N_HEAD      = 4
N_LAYER     = 8
SEQ_LEN     = 512
BATCH_SIZE  = 32
LR          = 3e-3
STEP_BUDGET = 8000
EVAL_BS     = 32
MID_LAYER   = 4
LATE_LAYER  = 7
EVAL_LAYERS = [MID_LAYER, LATE_LAYER]
K_FIXED     = 4                         # fixed k so d=8 is comparable

for d in D_LIST:
    assert d % N_HEAD == 0 and (d // N_HEAD) % 2 == 0

n_models = len(SEEDS) * len(D_LIST) * (1 + len(KEY_DEPTHS))
print(f"grammar deg={MIN_DEGREE} NT={NT_SIZE} T={TERM_SIZE} | vocab={VOCAB_SIZE} | mix A={MIX_A}")
print(f"grid: d={D_LIST} x [control + key-depth {KEY_DEPTHS} @ rho={RHO}] x seeds={SEEDS} "
      f"= {n_models} models (~{n_models*2.5:.0f} min)")

grammar deg=4 NT=4 T=4 | vocab=6 | mix A=0.7
grid: d=[8, 32, 96] x [control + key-depth [1, 2] @ rho=0.5] x seeds=[0] = 9 models (~22 min)


## 2. Grammar, sampler, context keys, deterministic conflict, renderer

In [3]:
def build_grammar(seed):
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1):
        symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE))
    rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = (terminals if child_level == LEVELS + 1
                         else symbols_by_level[child_level])
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE)
            seen = set(); tries = 0
            while len(rules[sym]) < degree and tries < 5000:
                tries += 1
                rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen: continue
                seen.add(rhs); rules[sym].append(rhs)
    return rules, symbols_by_level

def sample_tree(rules, rng, symbol=(1, 0), level=1):
    rhs = rng.choice(rules[symbol])
    children = []
    for child_sym in rhs:
        if level == LEVELS:
            children.append(child_sym)
        else:
            children.append(sample_tree(rules, rng, symbol=child_sym, level=level + 1))
    return (symbol, children)

def leaves(tree):
    out = []
    def walk(n):
        (level, idx), ch = n
        if level == LEVELS:
            out.extend(ch)
        else:
            for c in ch: walk(c)
    walk(tree)
    return out

# ---- context keys: which (grandparent, parent, position) contexts actually occur ----
def reachable_symbols(rules):
    reach = {1: {(1, 0)}}
    for l in range(1, LEVELS):
        nxt = set()
        for s in reach[l]:
            for rhs in rules[s]:
                nxt.update(rhs)
        reach[l + 1] = nxt
    return reach

def enumerate_keys(rules, depth):
    reach = reachable_symbols(rules)
    keys = set()
    for gp in reach[LEVELS - 1]:
        for p in {c for rhs in rules[gp] for c in rhs}:
            for rhs6 in rules[p]:
                for k in range(len(rhs6)):
                    keys.add((gp[1], p[1], k) if depth == 2 else (p[1], k))
    return sorted(keys)

DERANGEMENTS = [q for q in itertools.permutations(range(TERM_SIZE))
                if all(q[i] != i for i in range(TERM_SIZE))]

def make_conflict(rules, depth, rho, seed):
    # Deterministic: the conflicted key set is a prefix of a fixed seeded ordering (nested in rho),
    # and each key owns a fixed derangement of the terminals. No per-draw randomness anywhere.
    keys = enumerate_keys(rules, depth)
    order = keys[:]
    random.Random(seed + 101).shuffle(order)
    n_conf = int(round((1.0 - rho) * len(keys)))
    conf_keys = set(order[:n_conf])
    drng = random.Random(seed + 202)
    perm_of = {key: DERANGEMENTS[drng.randrange(len(DERANGEMENTS))] for key in keys}
    ident = tuple(range(TERM_SIZE))
    isconf = [[[False] * MAXK for _ in range(NT_SIZE)] for _ in range(NT_SIZE)]
    perm   = [[[ident] * MAXK for _ in range(NT_SIZE)] for _ in range(NT_SIZE)]
    for gp in range(NT_SIZE):
        for p in range(NT_SIZE):
            for k in range(MAXK):
                key = (gp, p, k) if depth == 2 else (p, k)
                if key in conf_keys:
                    isconf[gp][p][k] = True
                    perm[gp][p][k] = perm_of[key]
    return dict(depth=depth, rho=rho, n_keys=len(keys), n_conf=n_conf,
                isconf=isconf, perm=perm, isconf_np=np.array(isconf, dtype=bool))

def render(tree, conflict):
    # Same word order as the anchor. Returns terminal tokens and, per token, its context (gp, p, k).
    toks, ctx = [], []
    isc, prm = conflict["isconf"], conflict["perm"]
    def walk(node, gp):
        (level, idx), children = node
        if level == LEVELS:
            for k, term in enumerate(children):
                out = prm[gp][idx][k][term] if isc[gp][idx][k] else term
                toks.append(out); ctx.append((gp, idx, k))
        else:
            nxt = idx if level == LEVELS - 1 else gp
            for c in children:
                walk(c, nxt)
    walk(tree, 0)
    return toks, ctx

substrate_seed = 0
rules, symbols_by_level = build_grammar(substrate_seed)
print(f"grammar: {sum(len(r) for r in rules.values())} rules over {len(rules)} NTs")
for dep in KEY_DEPTHS:
    cf = make_conflict(rules, dep, RHO, substrate_seed)
    print(f"key depth {dep}: {cf['n_keys']} reachable contexts, {cf['n_conf']} conflicted at rho={RHO}")

grammar: 84 rules over 21 NTs
key depth 1: 12 reachable contexts, 6 conflicted at rho=0.5
key depth 2: 45 reachable contexts, 22 conflicted at rho=0.5


## 3. Pre-flight (pure python, asserts)
Guards against the bug classes that hit earlier experiments: identity at rho=1, conflicts exactly
where intended, every conflicted token really differs, rendering is deterministic.

In [4]:
_rng = random.Random(5)
_trees = [sample_tree(rules, _rng) for _ in range(1500)]

# rho=1 control: B is token-for-token identical to A
c1 = make_conflict(rules, 1, 1.0, substrate_seed)
assert c1["n_conf"] == 0 and not c1["isconf_np"].any()
for t in _trees:
    b, _ = render(t, c1)
    assert b == leaves(t)
print("rho=1: L_B rendering == L_A rendering on all 1500 trees  OK")

for dep in KEY_DEPTHS:
    cf = make_conflict(rules, dep, RHO, substrate_seed)
    n_tok = n_conf_tok = 0
    for t in _trees:
        a = leaves(t); b, ctx = render(t, cf)
        assert len(a) == len(b)
        for x, y, (g, p, k) in zip(a, b, ctx):
            if cf["isconf"][g][p][k]:
                assert x != y            # derangement: every conflicted token differs
                n_conf_tok += 1
            else:
                assert x == y            # clean contexts untouched
            n_tok += 1
        b2, _ = render(t, cf)
        assert b2 == b                   # deterministic
    print(f"key depth {dep}: conflicts exactly at conflicted contexts  OK  "
          f"| token-level conflict rate = {n_conf_tok / n_tok:.3f}")

# nesting across rho (same ordering + same derangements)
for dep in KEY_DEPTHS:
    big   = make_conflict(rules, dep, 0.5, substrate_seed)["isconf_np"]
    small = make_conflict(rules, dep, 0.8, substrate_seed)["isconf_np"]
    assert (small <= big).all()
print("conflict sets nested across rho  OK")

rho=1: L_B rendering == L_A rendering on all 1500 trees  OK
key depth 1: conflicts exactly at conflicted contexts  OK  | token-level conflict rate = 0.493
key depth 2: conflicts exactly at conflicted contexts  OK  | token-level conflict rate = 0.548
conflict sets nested across rho  OK


## 4. TinyRotaryGPT (same as sanity)

In [5]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return ((q * cos) + (rotate_half(q) * sin),
            (k * cos) + (rotate_half(k) * sin))

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)
    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head; self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(),
                                 nn.Linear(4*n_embd, n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd,
                 n_layer=N_LAYER, n_head=N_HEAD, max_seq_len=SEQ_LEN + 1):
        super().__init__()
        assert n_embd % n_head == 0 and (n_embd // n_head) % 2 == 0
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks: x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

def count_params(d):
    return sum(p.numel() for p in TinyRotaryGPT(VOCAB_SIZE, n_embd=d).parameters())

print("params:", {d: f"{count_params(d):,}" for d in D_LIST})

params: {8: '7,088', 32: '102,080', 96: '896,064'}


## 5. Data
Anchor: fresh online trees every batch (no repetition). L_B: bounded pool of `N_B` trees (the data-budget
knob), the same trees in every cell so cells differ only in how L_B renders them.
Eval: held-out trees (own seed, never in any pool), rendered in both languages; validity is an explicit mask.

In [6]:
def online_batch_A(rng):
    target = BATCH_SIZE * (SEQ_LEN + 1)
    stream = []
    while len(stream) < target:
        stream.append(BOS["A"])
        stream.extend(leaves(sample_tree(rules, rng)))
    return torch.tensor(stream[:target], dtype=torch.long).view(BATCH_SIZE, SEQ_LEN + 1)

def build_pool_B(trees, conflict):
    stream = []
    for t in trees:
        toks, _ = render(t, conflict)
        stream.append(BOS["B"]); stream.extend(toks)
    s = torch.tensor(stream, dtype=torch.long)
    n = len(s) // (SEQ_LEN + 1)
    assert n > 0
    return s[: n * (SEQ_LEN + 1)].view(n, SEQ_LEN + 1)

def build_eval(trees, conflict):
    # numpy arrays; identity word order => position j in the A row matches position j in the B row
    rowsA, rowsB, ctxs = [], [], []
    for t in trees:
        a = leaves(t); b, ctx = render(t, conflict)
        rowsA.append([BOS["A"]] + a)
        rowsB.append([BOS["B"]] + b)
        ctxs.append([(-1, -1, -1)] + ctx)
    maxlen = max(len(r) for r in rowsA)
    assert maxlen <= SEQ_LEN + 1, f"eval tree of {maxlen} tokens exceeds context"
    n = len(rowsA)
    idxA = np.zeros((n, maxlen), np.int64); idxB = np.zeros((n, maxlen), np.int64)
    valid = np.zeros((n, maxlen), bool)
    ctx_arr = -np.ones((n, maxlen, 3), np.int64)
    for i in range(n):
        L = len(rowsA[i])
        idxA[i, :L] = rowsA[i]; idxB[i, :L] = rowsB[i]; valid[i, :L] = True
        ctx_arr[i, :L] = np.array(ctxs[i])
    return dict(idxA=idxA, idxB=idxB, valid=valid,
                gp=ctx_arr[..., 0], p=ctx_arr[..., 1], k=ctx_arr[..., 2])

def position_masks(ev, ref_isconf_np):
    # masks over TARGET positions (y = row[1:]); a target is 'conflicted' if its context is in ref set
    valid_y = np.ascontiguousarray(ev["valid"][:, 1:])
    gp = ev["gp"][:, 1:]; p = ev["p"][:, 1:]; k = ev["k"][:, 1:]
    conf = np.zeros(valid_y.shape, bool)
    conf[valid_y] = ref_isconf_np[gp[valid_y], p[valid_y], k[valid_y]]
    clean = valid_y & ~conf
    return valid_y, np.ascontiguousarray(conf), np.ascontiguousarray(clean)

def check_alignment(ev, conflict):
    # With ref == the rendering's own conflict set, 'conflicted' targets must be EXACTLY the
    # positions where the A and B rows hold different tokens. Catches any off-by-one / key bug.
    valid_y, conf, clean = position_masks(ev, conflict["isconf_np"])
    differs = (ev["idxA"][:, 1:] != ev["idxB"][:, 1:]) & valid_y
    assert (differs == conf).all(), "eval mask misaligned with actual conflicts"
    return conf.sum() / valid_y.sum()

def sample_eval_trees(seed):
    rng = random.Random(seed + 31337)
    out = []
    while len(out) < N_EVAL:
        t = sample_tree(rules, rng)
        if len(leaves(t)) <= SEQ_LEN - 2:
            out.append(t)
    return out

# local sanity of the eval builder on the grammar
_ev = build_eval(sample_eval_trees(0)[:50], make_conflict(rules, 2, RHO, substrate_seed))
_frac = check_alignment(_ev, make_conflict(rules, 2, RHO, substrate_seed))
print(f"eval alignment check OK (conflicted target fraction {_frac:.3f}); eval max len {_ev['idxA'].shape[1]}")

eval alignment check OK (conflicted target fraction 0.550); eval max len 405


## 6. Metrics: held-out CE and feature-space principal angles, split by conflicted / clean targets

In [7]:
def top_subspace(X, k):
    Xc = (X - X.mean(0, keepdim=True)).double()
    evals, evecs = torch.linalg.eigh(Xc.T @ Xc)       # feature-space (d x d) covariance
    return evecs[:, -k:]                               # top-k principal directions, d x k

def subspace_angle(X, Y, k):
    k = min(k, X.shape[1])
    s = torch.linalg.svdvals(top_subspace(X, k).T @ top_subspace(Y, k)).clamp(0, 1)
    return torch.rad2deg(torch.arccos(s)).mean().item()

@torch.no_grad()
def evaluate(model, ev, ref_isconf_np):
    model.eval()
    valid_y, conf, clean = position_masks(ev, ref_isconf_np)
    valid_t, conf_t, clean_t = (torch.from_numpy(m) for m in (valid_y, conf, clean))
    n, T = ev["idxA"].shape
    d = model.tok_emb.embedding_dim
    res = {"conf_frac": float(conf.sum()) / float(valid_y.sum())}
    H = {}
    for name in ("A", "B"):
        idx = torch.from_numpy(ev["idx" + name])
        ce = torch.zeros(n, T - 1)
        hs = {l: torch.zeros(n, T - 1, d) for l in EVAL_LAYERS}
        for i in range(0, n, EVAL_BS):
            b = idx[i:i + EVAL_BS].to(DEVICE)
            x, y = b[:, :-1], b[:, 1:]
            cap = {}
            hooks = [model.blocks[l].register_forward_hook(
                         lambda m, inp, out, l=l: cap.__setitem__(l, out.detach().float().cpu()))
                     for l in EVAL_LAYERS]
            with autocast_ctx():
                logits, _ = model(x)
            for h in hooks: h.remove()
            per_tok = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)),
                                      y.reshape(-1), reduction="none").view(len(b), -1)
            ce[i:i + len(b)] = per_tok.cpu()
            for l in EVAL_LAYERS:
                hs[l][i:i + len(b)] = cap[l]
        res[f"ce_{name}_all"]   = ce[valid_t].mean().item()
        res[f"ce_{name}_conf"]  = ce[conf_t].mean().item() if conf.any() else float("nan")
        res[f"ce_{name}_clean"] = ce[clean_t].mean().item()
        H[name] = {l: (hs[l][conf_t], hs[l][clean_t]) for l in EVAL_LAYERS}
    for l in EVAL_LAYERS:
        res[f"ang_conf_L{l}"]  = (subspace_angle(H["A"][l][0], H["B"][l][0], K_FIXED)
                                  if conf.any() else float("nan"))
        res[f"ang_clean_L{l}"] = subspace_angle(H["A"][l][1], H["B"][l][1], K_FIXED)
    return res

print("metrics ready")

metrics ready


## 7. Train + sweep
Per (seed, d): one **control** model (rho=1, L_B identical to L_A) and one model per key depth at rho=0.5.
Same init, same anchor batches, same language order, same L_B trees across all of them, so differences
come only from how L_B renders. The control is evaluated under each depth's conflict set, which gives the
floor for every split.

In [8]:
def train_cell(d, conflict, trees_B, seed, steps=STEP_BUDGET):
    torch.manual_seed(seed)
    model = TinyRotaryGPT(VOCAB_SIZE, n_embd=d).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    pool = build_pool_B(trees_B, conflict).to(DEVICE)
    rngA = random.Random(seed + 1000)
    rngM = random.Random(seed + 2000)
    model.train(); losses = []
    t0 = time.time()
    for step in range(steps):
        if rngM.random() < MIX_A:
            batch = online_batch_A(rngA).to(DEVICE)
        else:
            batch = pool[torch.randint(0, pool.shape[0], (BATCH_SIZE,), device=DEVICE)]
        x, y = batch[:, :-1], batch[:, 1:]
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        losses.append(loss.item())
    return model, losses, time.time() - t0

results = {}
t_all = time.time()
for seed in SEEDS:
    rng_pool = random.Random(seed + 3000)
    trees_B = [sample_tree(rules, rng_pool) for _ in range(N_B)]
    eval_trees = sample_eval_trees(seed)
    refs = {dep: make_conflict(rules, dep, RHO, substrate_seed) for dep in KEY_DEPTHS}
    ctrl = make_conflict(rules, 1, 1.0, substrate_seed)

    for d in D_LIST:
        # ---- control (rho=1) ----
        print(f"\n[seed={seed} d={d}] control (rho=1)")
        model, losses, secs = train_cell(d, ctrl, trees_B, seed)
        ev = build_eval(eval_trees, ctrl)
        assert (ev["idxA"][:, 1:] == ev["idxB"][:, 1:]).all()      # control: identical text
        for dep in KEY_DEPTHS:
            results[("ctrl", seed, d, dep)] = evaluate(model, ev, refs[dep]["isconf_np"])
        r = results[("ctrl", seed, d, KEY_DEPTHS[0])]
        print(f"  train {secs/60:.1f} min  tail loss {_st.mean(losses[-100:]):.4f}  "
              f"CE_A={r['ce_A_all']:.4f} CE_B={r['ce_B_all']:.4f}")
        del model
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

        # ---- conflicted models, one per key depth ----
        for dep in KEY_DEPTHS:
            cf = refs[dep]
            print(f"[seed={seed} d={d}] key depth {dep}, rho={RHO} "
                  f"({cf['n_conf']}/{cf['n_keys']} contexts conflicted)")
            model, losses, secs = train_cell(d, cf, trees_B, seed)
            ev = build_eval(eval_trees, cf)
            frac = check_alignment(ev, cf)
            r = evaluate(model, ev, cf["isconf_np"])
            results[("conf", seed, d, dep)] = r
            print(f"  train {secs/60:.1f} min  tail loss {_st.mean(losses[-100:]):.4f}  "
                  f"CE_A={r['ce_A_all']:.4f} CE_B={r['ce_B_all']:.4f}  conflicted targets {frac:.3f}")
            del model
            if DEVICE.type == "cuda": torch.cuda.empty_cache()

print(f"\n[sweep done in {(time.time()-t_all)/60:.1f} min]")


[seed=0 d=8] control (rho=1)
  train 1.9 min  tail loss 1.0547  CE_A=1.0371 CE_B=1.0380
[seed=0 d=8] key depth 1, rho=0.5 (6/12 contexts conflicted)
  train 1.9 min  tail loss 1.1097  CE_A=1.0859 CE_B=1.1128  conflicted targets 0.494
[seed=0 d=8] key depth 2, rho=0.5 (22/45 contexts conflicted)
  train 1.9 min  tail loss 1.1523  CE_A=1.0544 CE_B=1.3214  conflicted targets 0.548

[seed=0 d=32] control (rho=1)
  train 1.8 min  tail loss 0.9763  CE_A=0.9627 CE_B=0.9627
[seed=0 d=32] key depth 1, rho=0.5 (6/12 contexts conflicted)
  train 1.8 min  tail loss 0.9941  CE_A=0.9781 CE_B=0.9881  conflicted targets 0.494
[seed=0 d=32] key depth 2, rho=0.5 (22/45 contexts conflicted)
  train 1.8 min  tail loss 1.0052  CE_A=0.9792 CE_B=1.0128  conflicted targets 0.548

[seed=0 d=96] control (rho=1)
  train 1.8 min  tail loss 0.9600  CE_A=0.9472 CE_B=0.9471
[seed=0 d=96] key depth 1, rho=0.5 (6/12 contexts conflicted)
  train 1.9 min  tail loss 1.0024  CE_A=0.9856 CE_B=0.9955  conflicted targets 0.

## 8. Summary

In [9]:
print("=" * 118)
print(f"OPTION B SMOKE  (deg={MIN_DEGREE}, rho={RHO}, k={K_FIXED}, mid=L{MID_LAYER}, late=L{LATE_LAYER})")
print("=" * 118)
print("conf-gap  = [CE_B - CE_A at conflicted targets](conflict model) - [same](rho=1 control)")
print("clean-gap = same, at clean targets (should stay ~0 if the conflict stays local)")
print("A-harm    = CE_A overall (conflict model) - CE_A overall (control): what the conflict costs the anchor")
print("angles    = within-model principal angle, A-run vs B-run hidden states, k=%d; floor = control model" % K_FIXED)
print("-" * 118)
print(f"{'seed':>4} {'key':>3} {'d':>4} | {'conf-gap':>9} {'clean-gap':>9} {'A-harm':>8} | "
      f"{'conf deg mid/late':>18} {'floor mid/late':>16} {'clean deg mid/late':>19} | {'conf%':>6}")
print("-" * 118)
for seed in SEEDS:
    for dep in KEY_DEPTHS:
        for d in D_LIST:
            c = results[("conf", seed, d, dep)]; z = results[("ctrl", seed, d, dep)]
            conf_gap  = (c["ce_B_conf"]  - c["ce_A_conf"])  - (z["ce_B_conf"]  - z["ce_A_conf"])
            clean_gap = (c["ce_B_clean"] - c["ce_A_clean"]) - (z["ce_B_clean"] - z["ce_A_clean"])
            a_harm = c["ce_A_all"] - z["ce_A_all"]
            m, l = MID_LAYER, LATE_LAYER
            print(f"{seed:>4} {dep:>3} {d:>4} | {conf_gap:>+9.4f} {clean_gap:>+9.4f} {a_harm:>+8.4f} | "
                  f"{c[f'ang_conf_L{m}']:>8.1f} /{c[f'ang_conf_L{l}']:>7.1f} "
                  f"{z[f'ang_conf_L{m}']:>8.1f} /{z[f'ang_conf_L{l}']:>5.1f} "
                  f"{c[f'ang_clean_L{m}']:>9.1f} /{c[f'ang_clean_L{l}']:>7.1f} | {100*c['conf_frac']:>5.1f}%")
    print("-" * 118)

print("\nRAW CE (held-out, all targets)")
print(f"{'key':>3} {'d':>4} | {'ctrl A':>8} {'ctrl B':>8} | {'conf A':>8} {'conf B':>8}")
for seed in SEEDS:
    for dep in KEY_DEPTHS:
        for d in D_LIST:
            c = results[("conf", seed, d, dep)]; z = results[("ctrl", seed, d, dep)]
            print(f"{dep:>3} {d:>4} | {z['ce_A_all']:>8.4f} {z['ce_B_all']:>8.4f} | "
                  f"{c['ce_A_all']:>8.4f} {c['ce_B_all']:>8.4f}")

print("""
HOW TO READ (conf-gap vs d, per key depth):
  gap shrinks as d grows, clean-gap ~0     -> conflict cost is capacity-gated and local: the N* shape
  gap flat across d                        -> irreducible floor (incl. any parse ambiguity the relabeling
                                              introduces) or not capacity-gated; the d-trend is the signal,
                                              not the level
  gap ~0 at every d                        -> too few lookups to bind capacity; compare key depth 2 vs 1
  gap huge at d=96, CE_B near chance       -> lookups not learnable at this width/steps
  conf angle >> floor while clean angle ~ floor -> separation is localized at the conflicts
  late-layer angle >> mid-layer angle      -> mostly output routing; weigh the mid layer
""")

OPTION B SMOKE  (deg=4, rho=0.5, k=4, mid=L4, late=L7)
conf-gap  = [CE_B - CE_A at conflicted targets](conflict model) - [same](rho=1 control)
clean-gap = same, at clean targets (should stay ~0 if the conflict stays local)
A-harm    = CE_A overall (conflict model) - CE_A overall (control): what the conflict costs the anchor
angles    = within-model principal angle, A-run vs B-run hidden states, k=4; floor = control model
----------------------------------------------------------------------------------------------------------------------
seed key    d |  conf-gap clean-gap   A-harm |  conf deg mid/late   floor mid/late  clean deg mid/late |  conf%
----------------------------------------------------------------------------------------------------------------------
   0   1    8 |   +0.0899   -0.0363  +0.0487 |     25.7 /   43.0      1.0 /  0.5      25.2 /   36.9 |  49.4%
   0   1   32 |   +0.0572   -0.0362  +0.0154 |     69.2 /   49.5      0.5 /  0.1      68.3 /   47.6 |  49.4%
   0   

In [10]:
for dep in KEY_DEPTHS:
    for d in D_LIST:
        c, z = results[("conf", 0, d, dep)], results[("ctrl", 0, d, dep)]
        dl = {k: c["ce_" + k] - z["ce_" + k] for k in ("A_conf", "A_clean", "B_conf", "B_clean")}
        print(dep, d, {k: round(v, 3) for k, v in dl.items()})

1 8 {'A_conf': 0.057, 'A_clean': 0.041, 'B_conf': 0.147, 'B_clean': 0.004}
1 32 {'A_conf': 0.018, 'A_clean': 0.013, 'B_conf': 0.075, 'B_clean': -0.023}
1 96 {'A_conf': 0.04, 'A_clean': 0.037, 'B_conf': 0.105, 'B_clean': -0.008}
2 8 {'A_conf': 0.013, 'A_clean': 0.022, 'B_conf': 0.262, 'B_clean': 0.309}
2 32 {'A_conf': 0.013, 'A_clean': 0.021, 'B_conf': 0.03, 'B_clean': 0.075}
2 96 {'A_conf': 0.021, 'A_clean': 0.027, 'B_conf': 0.022, 'B_clean': 0.072}


In [11]:
def online_batch_B(conflict, rng):
    target = BATCH_SIZE * (SEQ_LEN + 1)
    stream = []
    while len(stream) < target:
        toks, _ = render(sample_tree(rules, rng), conflict)
        stream.append(BOS["B"]); stream.extend(toks)
    return torch.tensor(stream[:target], dtype=torch.long).view(BATCH_SIZE, SEQ_LEN + 1)

def train_B_only(d, conflict, seed, steps=STEP_BUDGET):
    torch.manual_seed(seed)
    model = TinyRotaryGPT(VOCAB_SIZE, n_embd=d).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    rng = random.Random(seed + 4000)
    model.train()
    for _ in range(steps):
        batch = online_batch_B(conflict, rng).to(DEVICE)
        x, y = batch[:, :-1], batch[:, 1:]
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return model

ded = {}
for dep in KEY_DEPTHS:
    cf = refs[dep]
    ev = build_eval(eval_trees, cf)
    for d in D_LIST:
        m = train_B_only(d, cf, 0)
        r = evaluate(m, ev, cf["isconf_np"])      # A-side fields are meaningless here; read ce_B_* only
        s = results[("conf", 0, d, dep)]
        print(dep, d, "shared-B minus dedicated-B:",
              {k: round(s["ce_B_" + k] - r["ce_B_" + k], 3) for k in ("conf", "clean", "all")})
        del m
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

1 8 shared-B minus dedicated-B: {'conf': 0.075, 'clean': 0.041, 'all': 0.058}
1 32 shared-B minus dedicated-B: {'conf': 0.044, 'clean': 0.018, 'all': 0.031}
1 96 shared-B minus dedicated-B: {'conf': 0.087, 'clean': 0.03, 'all': 0.058}
2 8 shared-B minus dedicated-B: {'conf': 0.151, 'clean': 0.129, 'all': 0.141}
2 32 shared-B minus dedicated-B: {'conf': 0.061, 'clean': 0.057, 'all': 0.059}
2 96 shared-B minus dedicated-B: {'conf': 0.059, 'clean': 0.057, 'all': 0.059}
